# RU-Guard Phase 4: QLoRA training and GGUF export (Colab or Kaggle, free T4)

Runbook: `docs/colab/04_train.md`. This notebook only installs, clones, mounts Drive and calls the repo scripts.
Run all cells in order. Every step is skipped when its output already exists, and training resumes from the last
epoch checkpoint, so after a disconnect just run all again. On Kaggle use the free GPU T4 x2 accelerator with Internet on.

In [ ]:
# 1. Environment: Colab (Drive) or Kaggle (/kaggle/working). One GPU only.
import os, subprocess
ON_KAGGLE = os.path.exists("/kaggle/working")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if ON_KAGGLE:
    BASE, REPO = "/kaggle/working", "/kaggle/working/ru-guard"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, REPO = "/content/drive/MyDrive/ru-guard", "/content/ru-guard"
RUNS, EXPORT, BUILD = f"{BASE}/runs", f"{BASE}/export", "/tmp/gguf_build"
os.makedirs(RUNS, exist_ok=True); os.makedirs(EXPORT, exist_ok=True)
print("kaggle" if ON_KAGGLE else "colab", RUNS, EXPORT)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. Helper: stream a shell command and stop on failure.
def run(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"failed with exit code {p.returncode}: {cmd}")

def done(path):
    return os.path.exists(path)

In [ ]:
# 3. Code: clone, or update an existing clone.
if done(REPO):
    run("git pull --ff-only")
else:
    run(f"git clone --depth 1 https://github.com/f243077-cell/Fine-Tuning-Project.git {REPO}", cwd="/")
run("git log --oneline -1")

In [ ]:
# 4. Training stack (versions checked from source, D45). Pip conflict warnings with other preinstalled packages are OK.
run('pip install -q "unsloth==2026.11.4" "unsloth_zoo==2026.11.4" "trl==1.15.0" "transformers==5.17.0" "peft==0.21.3" pyyaml')
run('pip list 2>/dev/null | grep -iE "^(unsloth|unsloth.zoo|trl|transformers|peft|torch|bitsandbytes|datasets|accelerate) "')

In [ ]:
# 5. Dry run: tokenize only. Expect train_rows 4378, train_dropped_long 0, optimizer_steps 411.
run("python -m training.train_qlora --config training/configs/qwen2.5-0.5b.yaml --dry-run")

In [ ]:
# 6. Train and export one model. Skips when the adapter or the export manifest already exists.
def train(config, name, *sets):
    if done(f"{RUNS}/{name}/adapter/adapter_config.json"):
        print(f"skip {name}: adapter exists"); return
    extra = " ".join(f"--set {s}" for s in sets)
    run(f"python -m training.train_qlora --config {config} --out {RUNS} --set run_name={name} {extra}")

def export(name):
    if done(f"{EXPORT}/{name}/export_manifest.json"):
        print(f"skip {name}: exported"); return
    run(f"python -m training.merge_and_export --adapter {RUNS}/{name}/adapter --name {name} --out {EXPORT} --build-dir {BUILD}")
    run(f"ls -lh {EXPORT}/{name}/gguf/")

train("training/configs/qwen2.5-0.5b.yaml", "ru-guard-0.5b")
export("ru-guard-0.5b")

In [ ]:
# 7. ru-guard-1.5b.
train("training/configs/qwen2.5-1.5b.yaml", "ru-guard-1.5b")
export("ru-guard-1.5b")

In [ ]:
# 8. Ablations on 0.5B (time-boxed; skip what does not fit).
train("training/configs/qwen2.5-0.5b.yaml", "ru-guard-0.5b-r8", "lora.r=8", "lora.alpha=16")
train("training/configs/qwen2.5-0.5b.yaml", "ru-guard-0.5b-no-en", "data.include_en=false")
train("training/configs/qwen2.5-0.5b.yaml", "ru-guard-0.5b-2ep", "train.epochs=2")

In [ ]:
# 9. Pack the small logs to bring back to the repo (training/reports/).
run("zip -qr train_logs.zip runs/*/train_log.csv runs/*/val_metrics.csv runs/*/run_manifest.json export/*/export_manifest.json && ls -lh train_logs.zip", cwd=BASE)